In [1]:
import os

from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks/Projek Fluency/Fluensy Model')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

## Data Extraction

In [ ]:

file_path = 'dataset-siap-traing-fix.csv'
try:
    df_raw = pd.read_csv(file_path)
    print(f"Berhasil membaca {len(df_raw)} baris data original dari CSV!")
except FileNotFoundError:
    print(f"Tolong upload file '{file_path}")
    raise

# GABUNGKAN TEXT (Biar AI NLP makin paham konteksnya)
df_raw['bio_text'] = df_raw['kategori_utama'].astype(str) + " | " + df_raw['niche_spesifik'].astype(str)

# IMPUTASI SINTETIS (The Magic Trick untuk IG Views)
def impute_views(row):
    # Jika IG views masih 0, asumsikan 15% dari followers
    if row['average_views'] == 0 and row['followers'] > 0:
        return int(row['followers'] * 0.15)
    return row['average_views']

df_raw['average_views'] = df_raw.apply(impute_views, axis=1)

# BUANG OUTLIER (Fokus ke Mikro-Makro, Abaikan Artis dengan Base Rate > 3 Miliar)
df_clean = df_raw[df_raw['base_rate'] <= 3000000000].copy()

# DUPLIKASI DATA (Augmentasi 20x agar AI punya cukup data untuk berlatih keras)
df_final = pd.concat([df_clean]*20, ignore_index=True) 
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Data siap disajikan ke TensorFlow: {len(df_final)} baris (Setelah Augmentasi).")

✅ Berhasil membaca 74 baris data original dari CSV!
✅ Data siap disajikan ke TensorFlow: 1400 baris (Setelah Augmentasi).


## Preprocessing untuk Tensorflow

In [4]:
# A. Otak Angka (3 Parameter)
scaler_X = MinMaxScaler()
X_num = scaler_X.fit_transform(df_final[['followers', 'engagement_rate', 'average_views']])

# B. Target AI (1 Parameter Mutlak: Base Rate)
scaler_Y = MinMaxScaler()
y_target = scaler_Y.fit_transform(df_final[['base_rate']])

# C. Otak Bahasa (Teks Gabungan)
X_text = df_final['bio_text'].astype(str).values

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_num, y_target, test_size=0.2, random_state=42
)

## Custom Layer

In [5]:
@tf.keras.utils.register_keras_serializable() 
class FluensyAttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(FluensyAttentionLayer, self).__init__(**kwargs)
    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1],), initializer='random_normal', trainable=True)
    def call(self, inputs):
        return inputs * tf.nn.sigmoid(self.w)

## Arsitektue Tensorflow Funcional API (2 Input, 1 Output)

In [6]:
max_vocab = 3000
max_len = 25 

# Jalur 1: Memproses Teks Niche (NLP)
text_input = layers.Input(shape=(1,), dtype=tf.string, name='input_text')
vectorizer = layers.TextVectorization(max_tokens=max_vocab, output_sequence_length=max_len)
vectorizer.adapt(X_text_train) 
x_text = vectorizer(text_input)
x_text = layers.Embedding(input_dim=max_vocab, output_dim=64)(x_text)
x_text = layers.GlobalAveragePooling1D()(x_text)
x_text = layers.Dense(32, activation='relu')(x_text)

# Jalur 2: Memproses Angka (Followers, ER, Views)
num_input = layers.Input(shape=(3,), name='input_numeric')
x_num = layers.Dense(64, activation='relu')(num_input)
x_num = layers.Dense(32, activation='relu')(x_num)

# Pertemuan 2 Jalur (Penggabungan Otak)
merged = layers.Concatenate()([x_text, x_num])
attended = FluensyAttentionLayer()(merged)

x_shared = layers.Dense(128, activation='relu')(attended)
x_shared = layers.Dropout(0.2)(x_shared)
x_shared = layers.Dense(64, activation='relu')(x_shared)

# Kepala Output: Base Rate
output_base = layers.Dense(1, activation='linear', name='out_base')(x_shared)

model_pricer = Model(inputs=[text_input, num_input], outputs=output_base)

## Compile & Training

In [ ]:
model_pricer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='huber', # Tahan terhadap outlier yang terlewat
    metrics=['mae']
)

# Callbacks Sakti agar Training Berhenti Saat Sudah Pintar
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, mode='min')
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, mode='min', verbose=1)

print("\nMemulai Proses Belajar (Training) The Pricer V3...\n")
history = model_pricer.fit(
    {'input_text': X_text_train, 'input_numeric': X_num_train},
    y_train,
    validation_data=({'input_text': X_text_test, 'input_numeric': X_num_test}, y_test),
    epochs=150, # Ditingkatkan agar maksimal
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1 # Tampilkan progress bar (Penting di Colab)
)



🚀 Memulai Proses Belajar (Training) The Pricer V3...

Epoch 1/150
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0184 - mae: 0.1124 - val_loss: 0.0174 - val_mae: 0.1443 - learning_rate: 0.0010
Epoch 2/150
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0136 - mae: 0.0885 - val_loss: 0.0105 - val_mae: 0.0582 - learning_rate: 0.0010
Epoch 3/150
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0111 - mae: 0.0711 - val_loss: 0.0078 - val_mae: 0.0470 - learning_rate: 0.0010
Epoch 4/150
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0091 - mae: 0.0623 - val_loss: 0.0057 - val_mae: 0.0422 - learning_rate: 0.0010
Epoch 5/150
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0081 - mae: 0.0571 - val_loss: 0.0052 - val_mae: 0.0395 - learning_rate: 0.0010
Epoch 6/150
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0079 - mae: 0.0550 - val_loss: 0.0063 - val_mae: 0.0451 - learning_rate: 0.0010
Epoch 7/150
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0072 - mae: 0.0508 - val_loss: 0.0045

## Evaluasi Mae

In [ ]:
eval_results = model_pricer.evaluate(
    {'input_text': X_text_test, 'input_numeric': X_num_test},
    y_test,
    verbose=0
)

mae_base = eval_results[1]

print(f"\n=============================================")
print(f"MAE BASE RATE SCALED : {mae_base:.4f} (Syarat Kelulusan Dicoding: < 0.02)")
print(f"=============================================")

if mae_base < 0.02:
    print("MODEL LOLOS STANDAR DICODING!")
else:
    print("MAE masih agak tinggi.")





🎯 MAE BASE RATE SCALED : 0.0182 (Syarat Kelulusan Dicoding: < 0.02)
🏆 SELAMAT BOS! MODEL ANDA LOLOS STANDAR DICODING!


## Export Model

In [ ]:
model_pricer.save('fluensy_pricer_v3.keras')
print("\n💾 Model AI The Pricer disave dengan nama 'fluensy_pricer_v3.keras")


💾 Model AI The Pricer disave dengan nama 'fluensy_pricer_v3.keras'. Silakan download!


## Inference Model

In [ ]:
print("\nSIMULASI THE MULTIPLIER ENGINE (ARSITEKTUR FINAL):")
# Anggap saja ada influencer Beauty dengan 1 Juta Followers, ER 3.5%, Views 150Ribu
test_niche = tf.constant(["Beauty & Fashion | Skincare; Makeup & Cosmetics"], dtype=tf.string)
test_stats = scaler_X.transform([[1000000, 3.5, 150000]]) # [Followers, ER, Views]

pred_base = model_pricer.predict({'input_text': test_niche, 'input_numeric': test_stats}, verbose=0)

# Mengembalikan skala angka asli ke Rupiah
base_rate_raw = scaler_Y.inverse_transform(pred_base)[0][0]
base_rate = max(100000, int(base_rate_raw)) # Safety minimal 100 Ribu

print(f"\n[AI PREDICTION]")
print(f"-> Prediksi Base Rate Universal : Rp {base_rate:,}")

print(f"\n[HASIL PERHITUNGAN MULTIPLIER (Hitungan Backend)]")
print("Jika Platform = 'instagram':")
print(f"  - base_rate (Reels 100%) : Rp {base_rate:,}")
print(f"  - story_rate (25%)       : Rp {int(base_rate * 0.25):,}")
print(f"  - post_rate (70%)        : Rp {int(base_rate * 0.70):,}")
print(f"  - pp_rate (15%)          : Rp {int(base_rate * 0.15):,}")
print(f"  * addon_owning (50%)     : Rp {int(base_rate * 0.50):,}")
print(f"  * addon_boost (30%)      : Rp {int(base_rate * 0.30):,}")
print(f"  * addon_link (40%)       : Rp {int(base_rate * 0.40):,}")

print("\nJika Platform = 'tiktok':")
tiktok_base = int(base_rate * 1.20)
print(f"  - base_rate (Video 120%) : Rp {tiktok_base:,}")
print(f"  - story_rate (30%)       : Rp {int(base_rate * 0.30):,}")
print(f"  - post_rate (80%)        : Rp {int(base_rate * 0.80):,}")
print(f"  - pp_rate (20%)          : Rp {int(tiktok_base * 0.20):,}")
print(f"  * addon_owning (50%)     : Rp {int(tiktok_base * 0.50):,}")
print(f"  * addon_boost (30%)      : Rp {int(tiktok_base * 0.30):,}")
print(f"  * addon_link (50%)       : Rp {int(tiktok_base * 0.50):,}")


🔍 SIMULASI THE MULTIPLIER ENGINE (ARSITEKTUR FINAL):

[AI PREDICTION]
-> Prediksi Base Rate Universal : Rp 12,940,336

[HASIL PERHITUNGAN MULTIPLIER (Hitungan Backend)]
Jika Platform = 'instagram':
  - base_rate (Reels 100%) : Rp 12,940,336
  - story_rate (25%)       : Rp 3,235,084
  - post_rate (70%)        : Rp 9,058,235
  - pp_rate (15%)          : Rp 1,941,050
  * addon_owning (50%)     : Rp 6,470,168
  * addon_boost (30%)      : Rp 3,882,100
  * addon_link (40%)       : Rp 5,176,134

Jika Platform = 'tiktok':
  - base_rate (Video 120%) : Rp 15,528,403
  - story_rate (30%)       : Rp 3,882,100
  - post_rate (80%)        : Rp 10,352,268
  - pp_rate (20%)          : Rp 3,105,680
  * addon_owning (50%)     : Rp 7,764,201
  * addon_boost (30%)      : Rp 4,658,520
  * addon_link (50%)       : Rp 7,764,201


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
